# STEP 5 — Analysis & Paper Figures

**What this step does:**
- Loads all result CSVs from steps 2–4
- Generates all 4 paper figures + a summary table
- Interprets results and flags whether the main hypothesis is confirmed

**Figures produced:**
- `figures/fig1_accuracy_scaling.png` — Accuracy vs model size, CoT vs Direct (3 datasets)
- `figures/fig2_ece_scaling.png` — ECE vs model size, CoT vs Direct
- `figures/fig3_reliability_diagrams.png` — Reliability diagrams for key models
- `figures/fig4_consistency_delta.png` — Consistency delta (CoT - Direct) vs model size
- `figures/fig5_combined_trustworthiness.png` — Combined heatmap: 3 axes × 7 models

**Prerequisite:** Steps 2, 3, 4 must all be complete

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '/kaggle/working/light_paper')
from src.session_init import init
P = init()
from src.metrics import (
    compute_ece, compute_brier, compute_auroc, compute_aurc,
    compute_reliability_summary, compute_divergence,
    compute_reliability_frontier, compute_scaling_trend,
    compute_reliability_diagram_data
)

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'figure.dpi': 150, 'lines.linewidth': 2})

acc_df  = pd.read_csv(P['results'] / 'accuracy_raw.csv')
cal_df  = pd.read_csv(P['results'] / 'calibration_raw.csv')
cons_df = pd.read_csv(P['results'] / 'consistency_raw.csv')
corr_df = pd.read_csv(P['results'] / 'confidence_instability_corr.csv')

rel_df = cal_df.merge(cons_df[['id','model','mode','dataset','consistency']],
                      on=['id','model','mode','dataset'], how='left')
rel_df['consistency'] = rel_df['consistency'].fillna(0.5)

summary = compute_reliability_summary(rel_df)
summary = compute_divergence(summary)
summary = compute_reliability_frontier(summary)
scaling = compute_scaling_trend(summary)

summary.to_csv(P['results'] / 'reliability_summary.csv', index=False)
print(summary[['model','size_b','mode','dataset','accuracy','ece','brier',
               'auroc','reliability_score','divergence']].sort_values(
               ['dataset','mode','size_b']).to_string(index=False))

## Figure 1 — Accuracy vs Model Size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors  = {'direct': '#1f77b4', 'cot': '#d62728'}
markers = {'direct': 'o',       'cot': 's'}

for ax, ds in zip(axes, ['gsm8k','bbh','truthfulqa']):
    for mode in ['direct','cot']:
        sub = summary[(summary['dataset']==ds)&(summary['mode']==mode)].sort_values('size_b')
        ax.plot(sub['size_b'], sub['accuracy'], color=colors[mode], marker=markers[mode], label=mode.upper())
    ax.set_xlabel('Model Size (B)'); ax.set_ylabel('Accuracy')
    ax.set_title(ds.upper()); ax.set_xscale('log')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Figure 1: Accuracy vs Model Size — CoT vs Direct', fontsize=13)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig1_accuracy_scaling.png', bbox_inches='tight')
plt.show(); print('✓ fig1 saved')

## Figure 2 — ECE (Calibration) vs Model Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ds in zip(axes, ['gsm8k', 'truthfulqa']):
    subset = ece_df[ece_df.dataset == ds].sort_values('size_b')
    for mode in ['direct', 'cot']:
        m_data = subset[subset.mode == mode]
        ax.plot(
            m_data['size_b'], m_data['ece'],
            color=COLORS[mode], marker=MARKERS[mode],
            label=mode.capitalize()
        )

    ax.set_title(DATASET_LABELS[ds])
    ax.set_xlabel('Model size (B params)')
    ax.set_ylabel('ECE (lower = better calibrated)')
    ax.axvline(x=3, color='gray', linestyle='--', alpha=0.4, label='3B threshold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Figure 2: Expected Calibration Error (Direct vs CoT)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_ece_scaling.png', bbox_inches='tight')
plt.show()
print('✓ fig2 saved')

## Figure 3 — Reliability Diagrams (for representative models)

In [ ]:
metrics_to_plot = ['ece','brier','auroc','mean_consistency']
metric_labels   = ['ECE ↓','Brier ↓','AUROC ↑','Consistency ↑']
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, metric, label in zip(axes.flat, metrics_to_plot, metric_labels):
    for mode, color in [('direct','#1f77b4'),('cot','#d62728')]:
        for ds, ls in [('gsm8k','-'),('bbh','--'),('truthfulqa',':')]:
            sub = summary[(summary['mode']==mode)&(summary['dataset']==ds)].sort_values('size_b')
            ax.plot(np.log(sub['size_b']), sub[metric], color=color, linestyle=ls,
                    marker='o', label=f'{mode}/{ds}', alpha=0.8)
    ax.set_xlabel('log(Model Size B)'); ax.set_ylabel(label)
    ax.set_title(label); ax.grid(True, alpha=0.3)

axes[0,0].legend(fontsize=7)
fig.suptitle('Figure 2: Reliability Metrics vs Model Scale', fontsize=13)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig2_reliability_scaling.png', bbox_inches='tight')
plt.show(); print('✓ fig2 saved')

## Figure 4 — Consistency Delta (CoT − Direct) vs Model Size

In [ ]:
representative = ['Qwen2.5-0.5B','Qwen2.5-1.5B','Qwen2.5-3B','Llama-3.1-8B']
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for ax, model_name in zip(axes.flat, representative):
    sub = rel_df[(rel_df['model']==model_name)&(rel_df['mode']=='cot')&(rel_df['dataset']=='gsm8k')]
    if len(sub) == 0: continue
    conf = sub['confidence'].fillna(0.5).values
    corr = sub['correct'].astype(int).values
    diag = compute_reliability_diagram_data(conf, corr)
    ax.bar(diag['mean_conf'], diag['mean_acc'], width=0.08, alpha=0.7, color='#1f77b4')
    ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect')
    ax.set_title(f'{model_name}\nECE={compute_ece(conf,corr):.3f}')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

fig.suptitle('Figure 3: Reliability Diagrams (CoT / GSM8K)', fontsize=13)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig3_reliability_diagrams.png', bbox_inches='tight')
plt.show(); print('✓ fig3 saved')

## Figure 5 — Combined Trustworthiness Heatmap

In [ ]:
pivot = corr_df.pivot_table(index=['model','mode'], columns='dataset', values='spearman_r')
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)));  ax.set_yticklabels([f'{m}/{md}' for m,md in pivot.index], fontsize=8)
plt.colorbar(im, ax=ax, label='Spearman r')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i,j]
        if not np.isnan(val): ax.text(j,i,f'{val:.2f}',ha='center',va='center',fontsize=7)
ax.set_title('Figure 4: Confidence → Paraphrase Instability (Spearman r)')
plt.tight_layout()
plt.savefig(P['figures'] / 'fig4_confidence_instability.png', bbox_inches='tight')
plt.show(); print('✓ fig4 saved')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ds_colors = {'gsm8k':'#1f77b4','bbh':'#d62728','truthfulqa':'#2ca02c'}

for ax, mode in zip(axes, ['direct','cot']):
    sub = summary[summary['mode']==mode]
    for ds, color in ds_colors.items():
        pts = sub[sub['dataset']==ds]
        ax.scatter(pts['accuracy'], pts['reliability_score'],
                   c=color, s=pts['size_b']*30, alpha=0.7, label=ds, zorder=3)
        for _, row in pts.iterrows():
            ax.annotate(row['model'].replace('Qwen2.5-','Q').replace('Llama-3.','L').replace('Gemma-2-','G'),
                        (row['accuracy'], row['reliability_score']), fontsize=6, ha='left', va='bottom')
    pareto = sub[sub['pareto_dominant']].sort_values('accuracy')
    if len(pareto):
        ax.step(pareto['accuracy'], pareto['reliability_score'],
                where='post', color='black', linewidth=1.5, linestyle='--', label='Pareto')
    ax.set_xlabel('Accuracy'); ax.set_ylabel('Reliability Score')
    ax.set_title(f'Mode: {mode.upper()}'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Figure 5: Accuracy–Reliability Pareto Frontier (point size = model size)', fontsize=13)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig5_pareto_frontier.png', bbox_inches='tight')
plt.show(); print('✓ fig5 saved')

## Hypothesis Test — Is the main claim confirmed?

In [ ]:
print('=== HYPOTHESIS VERIFICATION ===\n')

high_div = summary[summary['divergence'].abs() > 1]
print(f'H0 — Divergence exists: {len(high_div)>0} ({len(high_div)} combos with |divergence|>1)')

cot_rel    = summary[summary['mode']=='cot']['reliability_score'].mean()
direct_rel = summary[summary['mode']=='direct']['reliability_score'].mean()
cot_acc    = summary[summary['mode']=='cot']['accuracy'].mean()
direct_acc = summary[summary['mode']=='direct']['accuracy'].mean()
print(f'H1 — CoT reliability gain ({cot_rel-direct_rel:+.3f}) > accuracy gain ({cot_acc-direct_acc:+.3f}): {(cot_rel-direct_rel)>(cot_acc-direct_acc)}')

small  = summary[summary['size_b'] <= 1.5]
medium = summary[summary['size_b'] >  1.5]
print(f'H2 — Scale reliability gap ({medium["reliability_score"].mean()-small["reliability_score"].mean():+.3f}) > accuracy gap ({medium["accuracy"].mean()-small["accuracy"].mean():+.3f}): {(medium["reliability_score"].mean()-small["reliability_score"].mean())>(medium["accuracy"].mean()-small["accuracy"].mean())}')

sig_neg = corr_df[(corr_df['spearman_r']<0) & (corr_df['p_value']<0.05)]
print(f'H3 — Confidence predicts instability: {len(sig_neg)>0} ({len(sig_neg)}/{len(corr_df)} significant)')

print('\n✅ STEP 5 COMPLETE')